# Spark Connect on EMR-on-EC2 — Local IDE Demo

This notebook connects from your local machine to a Spark Connect session running on an Amazon EMR on EC2 cluster.

**Prereqs on your laptop:**
- Python 3.11 recommended (matches the EMR worker Python if you plan to use UDFs)
- Run Cell 0 below to install the pinned `boto3`/`botocore`/`pyspark[connect]` versions in this kernel
- AWS credentials configured for short-lived, least-privilege access (see Security considerations below)

**Set these environment variables before running Cell 1** (nothing infrastructure-specific is hardcoded in the notebook):
```bash
export EMR_CLUSTER_ID="j-XXXXXXXXXXXXX"
export EMR_RUNTIME_ROLE_ARN="arn:aws:iam::<account-id>:role/<runtime-role>"
export AWS_REGION="us-west-2"
```


## Security considerations

This demo is intentionally minimal. Before adapting it for shared or production use, apply the AWS Well-Architected Framework Security Pillar controls below.

**Identity & access management**
- Scope the **runtime role** (`EMR_RUNTIME_ROLE_ARN`) to only the S3 prefixes this job reads/writes and only the EMR session APIs it needs — avoid wildcard (`*`) actions and resources.
- Scope the **caller identity** to the minimum EMR permissions: `elasticmapreduce:StartSession`, `GetSession`, `GetSessionEndpoint`, `TerminateSession`, plus `iam:PassRole` restricted to the specific runtime role ARN.
- Restrict the runtime role's **trust policy** so only the intended principals can assume it.

**Credential hygiene**
- Use short-lived credentials from **AWS IAM Identity Center (SSO)** or an assume-role workflow. Do **not** store long-term IAM user access keys in `~/.aws/credentials`.
- Enforce **MFA** on the identity used to run this notebook.

**Authentication token handling**
- The `AuthToken` returned by `get_session_endpoint` is a short-lived bearer secret. Never print, log, or persist it. This notebook keeps it in a local variable only and does not echo it.
- Notebook cell outputs are saved to the `.ipynb` file. **Clear all outputs** (`Cell → All Output → Clear`) before saving or committing, and exclude notebooks with live session data from source control.
- If a token expires mid-session, re-call `get_session_endpoint` rather than caching or reusing a stale value.

**Data protection at rest**
- Enable **S3 Block Public Access** on the target bucket, turn on **default server-side encryption (SSE-KMS recommended)**, and attach a bucket policy that restricts access to the runtime role ARN. Consider versioning/Object Lock for integrity.

**Network protection**
- Deploy the EMR cluster in a VPC with security groups that allow the Spark Connect port (443) only from known corporate egress ranges.
- Prefer **AWS PrivateLink / VPN / Direct Connect** over traversing the public internet. `use_ssl=true` provides TLS in transit; also confirm your HTTP client validates the endpoint certificate.

**Detective controls**
- Enable **AWS CloudTrail** to capture EMR (`StartSession`, `GetSessionEndpoint`, `TerminateSession`) and S3 API activity, **S3 server access logging / CloudTrail data events** on the bucket, **CloudWatch Logs** for EMR application logs, and **Amazon GuardDuty** for anomaly detection.

**Supply chain**
- Cell 0 pins exact package versions. For shared/production use, install from a scanned private repository such as **AWS CodeArtifact**, use a lock file with `pip install --require-hashes`, and run a vulnerability scan (e.g., `pip-audit`) before use.

**Incident response & availability**
- If a session or the runtime role is compromised: terminate the session (Cell 4), revoke the role's active sessions (attach a deny-all policy / rotate), and alert your security team per the AWS Incident Response Guide.
- Always run the cleanup cell to release cluster resources, and configure an EMR session idle timeout so abandoned sessions self-terminate.

In [ ]:
# Cell 0 — Install pinned package versions in the current kernel.
# EMR emr.start_session / get_session / get_session_endpoint require botocore>=1.43.24.
# Exact pins give reproducible, auditable installs (SEC10). For shared/production use,
# install from AWS CodeArtifact and/or use `pip install --require-hashes` with a lock file,
# and scan with a tool such as `pip-audit` before running.
# Restart the kernel after this cell finishes (toolbar → Restart).
%pip install --quiet "boto3==1.43.63" "botocore==1.43.63" "pyspark[connect]==4.0.0"

import boto3, botocore, pyspark
print("boto3   :", boto3.__version__)
print("botocore:", botocore.__version__)
print("pyspark :", pyspark.__version__)
print("\n>>> Now restart the kernel (toolbar → Restart), then run Cell 1.")

In [ ]:
# Cell 1 — Start a Spark Connect session on the cluster (all in one cell)
import boto3, os, time, urllib.parse

# Read infrastructure identifiers from environment variables so cluster IDs, role ARNs,
# and account details are never hardcoded in the notebook (SEC02, avoids leaking identifiers
# if the notebook is shared or committed to source control).
try:
    CLUSTER_ID   = os.environ["EMR_CLUSTER_ID"]
    RUNTIME_ROLE = os.environ["EMR_RUNTIME_ROLE_ARN"]
except KeyError as missing:
    raise SystemExit(f"Set the required environment variable before running: {missing}. "
                     "See the prerequisites cell at the top of this notebook.")
REGION = os.environ.get("AWS_REGION", "us-west-2")

emr = boto3.client("emr", region_name=REGION)

# start session (per-user, Runtime-Role isolated)
sess = emr.start_session(
    ClusterId=CLUSTER_ID, Name="local-notebook-demo",
    ExecutionRoleArn=RUNTIME_ROLE,
)
session_id = sess["Id"]
print("session started")

# poll to IDLE (usually 30–60s)
for i in range(30):
    state = emr.get_session(ClusterId=CLUSTER_ID, SessionId=session_id)["Session"]["State"]
    print(f"  state={state}")
    if state == "IDLE": break
    if state in ("FAILED", "TERMINATED"): raise RuntimeError(f"session {state} before IDLE")
    time.sleep(10)

# get endpoint + auth token
ep = emr.get_session_endpoint(ClusterId=CLUSTER_ID, SessionId=session_id)
host = urllib.parse.urlparse(ep["Endpoint"]).netloc or ep["Endpoint"].replace("https://", "")

# Validate the endpoint host resolves to an expected AWS domain before building the
# connection URL. This guards against a spoofed/malformed API response redirecting the
# Spark Connect client to an attacker-controlled host (SSRF / endpoint spoofing).
if not host.endswith(".amazonaws.com"):
    raise ValueError(f"Unexpected Spark Connect endpoint host; refusing to connect: {host!r}")

# The AuthToken is a short-lived bearer secret. Keep it in a local variable only —
# never print, log, or persist it, and do not print token expiry metadata either.
token = ep["AuthToken"]
sc_url = (f"sc://{host}:443/"
          f";use_ssl=true"
          f";x-aws-proxy-auth={token}"
          f";authorization={session_id}")
print("session is IDLE; Spark Connect endpoint ready")

In [ ]:
# Cell 2 — Connect via Spark Connect and confirm
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(sc_url).getOrCreate()
print("Connected. Spark version:", spark.version)
spark.sql("SELECT 'Hello from EMR-on-EC2 via Spark Connect' AS msg, current_timestamp() AS ts").show(truncate=False)

Connected. Spark version: 4.0.2-amzn-0
+---------------------------------------+--------------------------+
|msg                                    |ts                        |
+---------------------------------------+--------------------------+
|Hello from EMR-on-EC2 via Spark Connect|2026-08-04 06:43:22.764915|
+---------------------------------------+--------------------------+



**S3 target security:** before running the round-trip below, ensure the target bucket has S3 Block Public Access enabled, default server-side encryption on (SSE-KMS recommended), and a bucket policy that restricts read/write to the runtime role ARN. The demo bucket name is a placeholder — replace it with your own secured bucket.

In [ ]:
# Cell 3 — DataFrame + S3 round-trip (cluster does the work, laptop shows the result)
import os
import pyspark.sql.functions as F

# Bucket/prefix come from an environment variable so no account-specific path is hardcoded.
path = os.environ.get("DEMO_S3_PATH", "s3://amzn-s3-demo-localnotebook/local-notebook-demo/")

df = (
    spark.range(0, 10_000)
    .withColumn("category", F.when(F.col("id") % 2 == 0, "even").otherwise("odd"))
)

df.groupBy("category").count().show()

df.write.mode("overwrite").parquet(path)
spark.read.parquet(path).filter("id < 20").orderBy("id").show(20)

+--------+-----+
|category|count|
+--------+-----+
|    even| 5000|
|     odd| 5000|
+--------+-----+



In [ ]:
# Cell 4 — Cleanup: terminate the session on the cluster (releases resources).
# Always run this. Consider also configuring an EMR session idle timeout so abandoned
# sessions self-terminate, and use terminate_session as the first step in incident response.
emr.terminate_session(ClusterId=CLUSTER_ID, SessionId=session_id)
print("terminated session")